model.parameters() 是 PyTorch 中 torch.nn.Module 类的一个核心方法。它返回一个迭代器（Iterator），用于访问模型中所有可学习的参数（<b>即权重 weights 和偏置 biases</b>）。
- 简单来说，<b>它是连接模型定义与优化器的桥梁。</b>

##  核心功能
当你定义了一个神经网络模型（继承自 nn.Module）后，模型内部包含了许多层（如 nn.Linear, nn.Conv2d 等）。这些层中包含需要训练的张量（Tensor）。
- 返回值：一个生成器（Generator），每次迭代返回一个 torch.Tensor（即参数本身）。
- 包含内容：<b>它会自动递归地查找模型中所有子模块的 requires_grad=True 的参数。</b>
- 不包含内容：它不包含参数的名称（如 "fc1.weight"），只包含参数张量本身。

## 最常见的用途：传递给优化器

这是 model.parameters() 90% 的使用场景。在训练开始前，<b>你需要告诉优化器（如 SGD, Adam）去更新哪些参数。</b>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. 定义模型
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 1)
    def forward(self, x):
        return self.fc(x)

model = SimpleNet()

# 2. 定义优化器
# 将 model.parameters() 传入，优化器就知道要更新哪些参数了
optimizer = optim.SGD(model.parameters(), lr=0.01)

# 3. 训练循环中的标准用法
# optimizer.zero_grad() -> loss.backward() -> optimizer.step()
# 其中 optimizer.step() 就是利用 model.parameters() 里的引用去更新数值的

## 其他用途

### 计算参数量：统计模型有多少个参数（例如计算 FLOPs 或模型大小时）。

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"总参数量：{total_params}")

### 冻结参数：虽然通常用 named_parameters() 更精准，但你也可以遍历 parameters() 来批量操作。

In [ ]:
# 冻结模型：将所有参数的梯度计算关闭
for param in model.parameters():
    param.requires_grad = False

### 查看参数统计信息：检查权重初始化是否正常，或者梯度是否消失/爆炸。

In [ ]:
for param in model.parameters():
    print(f"均值：{param.mean()},标准差：{param.std()")

##  关键对比：parameters() vs named_parameters() vs state_dict()

| 特性 | `model.parameters()` | `model.named_parameters()` | `model.state_dict()` |
| :--- | :--- | :--- | :--- |
| 返回类型 | 迭代器 (Iterator) | 迭代器 (Iterator) | 字典 (OrderedDict) |
| 包含名称 | ❌ 只有张量 | ✅ (名称, 张量) 元组 | ✅ 键是名称，值是张量 |
| 主要用途 | 优化器更新、参数统计 | 选择性冻结、打印特定层 | 保存模型、加载模型 |
| 内存占用 | 低 (引用) | 低 (引用) | 高 (复制数据) |
| 示例 | `optim.SGD(model.parameters())` | `for name, p in model.named_parameters():` | `torch.save(model.state_dict())` |



In [1]:
import torch
import torch.nn as nn

class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3)
        self.fc = nn.Linear(100, 10)

    def forward(self, x):
        return self.fc(x)

model = MyModel()

# --- 1. 使用 parameters() ---
print("--- 使用 parameters() ---")
# 只能看到张量数据，不知道是哪个层的
for p in model.parameters():
    print(f"参数形状: {p.shape}") 

# --- 2. 使用 named_parameters() ---
print("\n--- 使用 named_parameters() ---")
# 可以看到 "conv1.weight", "fc.bias" 等名字
for name, p in model.named_parameters():
    print(f"{name}: {p.shape}")

# --- 3. 使用 state_dict() ---
print("\n--- 使用 state_dict() ---")
# 这是一个字典，常用于保存
state = model.state_dict()
print(type(state)) # <class 'collections.OrderedDict'>

--- 使用 parameters() ---
参数形状: torch.Size([16, 3, 3, 3])
参数形状: torch.Size([16])
参数形状: torch.Size([10, 100])
参数形状: torch.Size([10])

--- 使用 named_parameters() ---
conv1.weight: torch.Size([16, 3, 3, 3])
conv1.bias: torch.Size([16])
fc.weight: torch.Size([10, 100])
fc.bias: torch.Size([10])

--- 使用 state_dict() ---
<class 'collections.OrderedDict'>


- 如果你要训练模型（初始化优化器），请用 model.parameters()。
- 如果你要打印模型结构或冻结特定层，请用 model.named_parameters()。
- 如果你要保存或加载模型权重，请用 model.state_dict()